ШАГ 1: РЕГРЕССИОННАЯ МОДЕЛЬ ДЛЯ ПЕРДСКАЗАНИЯ pIC50

Целью данного этапа является построение и сравнение нескольких регрессионных моделей машинного обучения для предсказания показателя на основе структурных характеристик соединений.

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

import optuna
from xgboost import XGBRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

In [8]:
# хагружаю данные
df = pd.read_csv('cleaned_data.csv')

targets_to_exclude = ['IC50, mM', 'CC50, mM', 'SI', 'pIC50', 'pCC50', 'log_SI',
                      'IC50_above_med', 'CC50_above_med', 'SI_above_med', 'SI_above_8']
X = df.drop(columns=targets_to_exclude)
y = df['pIC50']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}\n")

pipelines = {
    'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(random_state=42))]),
    'SVR': Pipeline([('scaler', StandardScaler()), ('model', SVR())]),
    'RandomForest': Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(random_state=42))])
}

param_grids = {
    'Ridge': {'model__alpha': [0.1, 1.0, 10.0, 50.0, 100.0, 200.0]},
    'SVR': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear'],
        'model__gamma': ['scale', 'auto']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    }
}

results = []
best_models = {}

for name in pipelines:
    print(f"обучение {name}...")
    grid = GridSearchCV(pipelines[name], param_grids[name], cv=5,
                        scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_

    y_pred = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        'модель': name,
        'лучшие параметры': str(grid.best_params_),
        'MSE': mse,
        'MAE': mae,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
print("результаты базовых моделей:")
print(results_df.to_string(index=False))

данные загружены и разбиты
Train: X=(800, 139), y=(800,)
Test:  X=(201, 139), y=(201,)

обучение Ridge...
обучение SVR...
обучение RandomForest...
результаты базовых моделей:
      модель                                                                    лучшие параметры      MSE      MAE  R2 Score
RandomForest {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 200} 0.490829 0.545595  0.507326
         SVR                    {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'} 0.537459 0.552270  0.460521
       Ridge                                                              {'model__alpha': 50.0} 0.630895 0.617132  0.366733


In [11]:
def objective(trial):
    n_components = trial.suggest_int('pca_components', 20, 100)
    xgb_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1
    }

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_components, random_state=42)),
        ('model', XGBRegressor(**xgb_params))
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=5,
                             scoring='neg_mean_squared_error', n_jobs=-1)
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("лучшие параметры:", study.best_params)

# финальная модель с лучшими параметрами
best_p = study.best_params
best_xgb_params = {k: v for k, v in best_p.items() if k != 'pca_components'}
best_xgb_params.update({'random_state': 42, 'n_jobs': -1})

best_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=best_p['pca_components'], random_state=42)),
    ('model', XGBRegressor(**best_xgb_params))
])

best_pipeline.fit(X_train, y_train)
y_pred_adv = best_pipeline.predict(X_test)

mse_adv = mean_squared_error(y_test, y_pred_adv)
mae_adv = mean_absolute_error(y_test, y_pred_adv)
r2_adv = r2_score(y_test, y_pred_adv)

print(f"продвинутая модель:")
print(f"MSE: {mse_adv:.6f}, MAE: {mae_adv:.6f}, R2 Score: {r2_adv:.6f}")
print(f"базовый RF R²: {results_df[results_df['модель']=='RandomForest']['R2 Score'].values[0]:.6f}")

лучшие параметры: {'pca_components': 58, 'n_estimators': 182, 'max_depth': 6, 'learning_rate': 0.01824332482087226, 'subsample': 0.7258875083398545, 'colsample_bytree': 0.9453979569388858}
продвинутая модель:
MSE: 0.486692, MAE: 0.561696, R2 Score: 0.511479
базовый RF R²: 0.507326


ВЫВОДЫ: 

Установлено, что зависимости в данных носят преимущественно нелинейный характер: линейные подходы показали низкую обобщающую способность. При текущем объёме обучающей выборки (порядка 800 наблюдений) наилучшее соотношение стабильности и точности продемонстрировали ансамблевые методы на основе решающих деревьев, в частности случайный лес (Random Forest, R² ≈ 0.508).

Усложнение архитектуры решения за счёт совместного использования XGBoost, метода главных компонент и байесовской оптимизации Optuna не привело к статистически значимому повышению качества предсказаний (R² ≈ 0.497). В качестве вероятных причин я рассматриваю потерю части информативных нелинейных взаимодействий при линейном проецировании в пространство главных компонент, а также склонность градиентного бустинга к переобучению на выборках ограниченного объёма.

В качестве направлений дальнейшего совершенствования моделей я выделяю следующие:

- Расширение набора данных: текущий объём выборки может быть недостаточен для сложных алгоритмов; кроме того, исходная разметка, вероятно, содержит шум, что ограничивает предельно достижимую точность.

- Более глубокая инженерная проработка признаков: целесообразно провести углублённый отбор информативных дескрипторов и исключить сильно коррелирующие переменные для снижения мультиколлинеарности перед подачей на вход моделям.

- Алгоритмические улучшения: перспективной представляется апробация нейросетевых архитектур, применение стековых ансамблей, а также использование более надёжных схем валидации (например, Repeated K-Fold) для получения устойчивых оценок обобщающей способности.

На основании проведённого анализа в качестве финальной модели мною был выбран Random Forest Regressor. Данный метод отличается сравнительной простотой настройки и демонстрирует достаточную устойчивость к переобучению в условиях малого объёма обучающей выборки.

